# Phase 6: Explainability & Model Understanding
**Legal Contract Analyzer (CUAD) | Anthony Rodrigues | 2026-04-19**

## Research Questions
1. **Feature utilization:** How many of 40K TF-IDF features does each LGBM actually use? Are we wasting capacity?
2. **N-gram decomposition:** Did trigrams actually contribute signal, or is it still mostly unigrams?
3. **TreeSHAP analysis:** What drives HIGH-RISK clause predictions? Do SHAP values validate legal domain knowledge?
4. **TP vs FP decomposition:** What features cause FALSE POSITIVES on high-risk clauses? Can we identify systematic failure patterns?
5. **Cross-clause feature sharing:** Do different clause detectors share features, or are they independent?

## What Mark already did (avoid duplication)
Mark's Phase 6 used the OLD pipeline (LGBM+LR blend, 20K features, Youden thresholds — test-fit, inflated metrics).
He analyzed LR coefficients, LGBM split-gain importance, and feature position in contracts.

**This notebook complements Mark's work by:**
- Using the CORRECTED Phase 5 champion (LGBM-only, 40K trigrams, class-prior thresholds)
- Running proper TreeSHAP (model-faithful explanations, not proxy importance)
- Decomposing by n-gram SIZE (new analysis)
- Comparing TP vs FP SHAP profiles (new analysis)
- Auditing feature utilization across 40K features (new analysis)

## Cell 1: Load data and saved model artifacts

In [1]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from src.feature_engineering import load_cuad_from_json, make_split, HIGH_RISK_CLAUSES

ROOT = Path('..')
RESULTS = ROOT / 'results'
MODELS = ROOT / 'models'

# Load data (same split as train.py)
df = load_cuad_from_json(ROOT / 'data' / 'raw' / 'CUADv1.json')
train_df, test_df, valid_clauses = make_split(df, test_size=0.2, seed=42)

# Load saved artifacts
vec = joblib.load(MODELS / 'vectorizer.joblib')
lgbm_models = joblib.load(MODELS / 'lgbm_models.joblib')
with open(MODELS / 'thresholds.json') as f:
    thresholds = json.load(f)
with open(MODELS / 'valid_clauses.json') as f:
    valid_clauses_saved = json.load(f)

# Verify clause alignment
assert valid_clauses == valid_clauses_saved, "Clause mismatch!"

# Transform
X_train = vec.transform(train_df['text'].values)
X_test = vec.transform(test_df['text'].values)
y_train = train_df[valid_clauses].values.astype(int)
y_test = test_df[valid_clauses].values.astype(int)

feature_names = np.array(vec.get_feature_names_out())
n_clauses = len(valid_clauses)
n_features = len(feature_names)
hr_valid = [c for c in HIGH_RISK_CLAUSES if c in valid_clauses]

print(f"Train: {X_train.shape[0]} contracts | Test: {X_test.shape[0]} contracts")
print(f"Features: {n_features:,} | Clauses: {n_clauses} | HIGH-RISK: {len(hr_valid)}")
print(f"HIGH-RISK clauses: {hr_valid}")
print(f"Models loaded: {sum(m is not None for m in lgbm_models)}/{n_clauses} fitted")

Train: 408 contracts | Test: 102 contracts
Features: 40,000 | Clauses: 28 | HIGH-RISK: 5
HIGH-RISK clauses: ['Uncapped Liability', 'IP Ownership Assignment', 'Change Of Control', 'Non-Compete', 'Liquidated Damages']
Models loaded: 28/28 fitted


## Cell 2: Feature Utilization Audit
**Hypothesis:** With max_depth=4 and 50 trees, each LGBM can use at most ~750 unique split features per clause. We have 40K features — most should be unused. If utilization is <5%, the model is naturally sparse and we're paying a vectorization cost for features it ignores.

In [2]:
utilization = {}
all_used_features = set()

for j, clause in enumerate(valid_clauses):
    m = lgbm_models[j]
    if m is None:
        continue
    imp = m.feature_importances_
    used = np.sum(imp > 0)
    util_pct = 100 * used / n_features
    used_idx = np.where(imp > 0)[0]
    all_used_features.update(used_idx)
    utilization[clause] = {
        'n_used': int(used),
        'pct': round(util_pct, 2),
        'n_positive_train': int(y_train[:, j].sum()),
        'top5': [(feature_names[i], float(imp[i])) for i in np.argsort(imp)[::-1][:5]],
    }

# Summary statistics
used_counts = [v['n_used'] for v in utilization.values()]
print(f"Feature utilization across {len(utilization)} clause models:")
print(f"  Mean features used per clause: {np.mean(used_counts):.0f} / {n_features:,} ({100*np.mean(used_counts)/n_features:.1f}%)")
print(f"  Min: {np.min(used_counts)} | Max: {np.max(used_counts)} | Median: {np.median(used_counts):.0f}")
print(f"  Union across ALL clauses: {len(all_used_features):,} / {n_features:,} ({100*len(all_used_features)/n_features:.1f}%)")
print(f"  UNUSED features (never split on by any clause): {n_features - len(all_used_features):,}")
print()

# Sort by utilization
sorted_util = sorted(utilization.items(), key=lambda x: x[1]['n_used'], reverse=True)
print(f"{'Clause':<40} {'Used':>6} {'%':>6} {'Train+':>7}")
print('-' * 62)
for clause, info in sorted_util:
    risk = 'HR' if clause in hr_valid else '  '
    print(f"{risk} {clause:<37} {info['n_used']:>6} {info['pct']:>5.1f}% {info['n_positive_train']:>6}")

Feature utilization across 28 clause models:
  Mean features used per clause: 277 / 40,000 (0.7%)
  Min: 219 | Max: 361 | Median: 276
  Union across ALL clauses: 4,886 / 40,000 (12.2%)
  UNUSED features (never split on by any clause): 35,114

Clause                                     Used      %  Train+
--------------------------------------------------------------
   Most Favored Nation                      361   0.9%     21
   No-Solicit Of Customers                  342   0.9%     26
   Non-Disparagement                        329   0.8%     31
   Third Party Beneficiary                  321   0.8%     23
   Non-Transferable License                 312   0.8%    116
   Post-Termination Services                297   0.7%    141
HR Uncapped Liability                       297   0.7%     87
   Minimum Commitment                       295   0.7%    132
   Volume Restriction                       294   0.7%     65
HR Liquidated Damages                       290   0.7%     47
   Cap On L

## Cell 3: N-gram Size Decomposition
**Hypothesis:** Trigrams (Phase 4 ablation winner over bigrams) should carry meaningful importance. If importance is >95% unigrams, the trigram expansion was unnecessary capacity that happened to help by including more bigrams.

In [3]:
# Classify each feature by n-gram size
ngram_sizes = np.array([len(f.split()) for f in feature_names])
uni_mask = ngram_sizes == 1
bi_mask = ngram_sizes == 2
tri_mask = ngram_sizes == 3

print(f"Feature count by n-gram size:")
print(f"  Unigrams: {uni_mask.sum():,} ({100*uni_mask.sum()/n_features:.1f}%)")
print(f"  Bigrams:  {bi_mask.sum():,} ({100*bi_mask.sum()/n_features:.1f}%)")
print(f"  Trigrams: {tri_mask.sum():,} ({100*tri_mask.sum()/n_features:.1f}%)")

# Per-clause importance by n-gram size
ngram_importance = {}
for j, clause in enumerate(valid_clauses):
    m = lgbm_models[j]
    if m is None:
        continue
    imp = m.feature_importances_
    total = imp.sum()
    if total == 0:
        continue
    uni_imp = imp[uni_mask].sum() / total
    bi_imp = imp[bi_mask].sum() / total
    tri_imp = imp[tri_mask].sum() / total
    ngram_importance[clause] = {
        'unigram_pct': round(100 * uni_imp, 1),
        'bigram_pct': round(100 * bi_imp, 1),
        'trigram_pct': round(100 * tri_imp, 1),
    }

# Aggregate
mean_uni = np.mean([v['unigram_pct'] for v in ngram_importance.values()])
mean_bi = np.mean([v['bigram_pct'] for v in ngram_importance.values()])
mean_tri = np.mean([v['trigram_pct'] for v in ngram_importance.values()])

print(f"\nMean importance by n-gram size (across {len(ngram_importance)} clauses):")
print(f"  Unigrams: {mean_uni:.1f}%")
print(f"  Bigrams:  {mean_bi:.1f}%")
print(f"  Trigrams: {mean_tri:.1f}%")

print(f"\nPer HIGH-RISK clause:")
print(f"{'Clause':<30} {'Unigram':>8} {'Bigram':>8} {'Trigram':>8}")
print('-' * 58)
for clause in hr_valid:
    if clause in ngram_importance:
        info = ngram_importance[clause]
        print(f"{clause:<30} {info['unigram_pct']:>7.1f}% {info['bigram_pct']:>7.1f}% {info['trigram_pct']:>7.1f}%")

Feature count by n-gram size:
  Unigrams: 5,678 (14.2%)
  Bigrams:  19,495 (48.7%)
  Trigrams: 14,827 (37.1%)

Mean importance by n-gram size (across 28 clauses):
  Unigrams: 39.0%
  Bigrams:  42.1%
  Trigrams: 18.9%

Per HIGH-RISK clause:
Clause                          Unigram   Bigram  Trigram
----------------------------------------------------------
Uncapped Liability                34.5%    41.6%    23.9%
IP Ownership Assignment           39.2%    43.6%    17.1%
Change Of Control                 31.7%    44.8%    23.5%
Non-Compete                       40.4%    43.7%    15.9%
Liquidated Damages                35.2%    45.9%    18.9%


## Cell 4: N-gram Decomposition Visualization

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: stacked bar per clause
ax = axes[0]
clauses_sorted = sorted(ngram_importance.keys(), key=lambda c: ngram_importance[c]['trigram_pct'], reverse=True)
uni_vals = [ngram_importance[c]['unigram_pct'] for c in clauses_sorted]
bi_vals = [ngram_importance[c]['bigram_pct'] for c in clauses_sorted]
tri_vals = [ngram_importance[c]['trigram_pct'] for c in clauses_sorted]
colors_hr = ['#d32f2f' if c in hr_valid else '#1976d2' for c in clauses_sorted]

x = range(len(clauses_sorted))
ax.bar(x, uni_vals, label='Unigram', color='#1565c0', alpha=0.85)
ax.bar(x, bi_vals, bottom=uni_vals, label='Bigram', color='#42a5f5', alpha=0.85)
ax.bar(x, tri_vals, bottom=[u+b for u,b in zip(uni_vals, bi_vals)], label='Trigram', color='#90caf9', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(clauses_sorted, rotation=90, fontsize=6)
ax.set_ylabel('Importance %')
ax.set_title('Feature Importance by N-gram Size per Clause')
ax.legend(loc='lower right')

# Right: aggregate pie
ax2 = axes[1]
sizes = [mean_uni, mean_bi, mean_tri]
labels = [f'Unigrams\n{mean_uni:.1f}%', f'Bigrams\n{mean_bi:.1f}%', f'Trigrams\n{mean_tri:.1f}%']
colors = ['#1565c0', '#42a5f5', '#90caf9']
ax2.pie(sizes, labels=labels, colors=colors, autopct='', startangle=90, textprops={'fontsize': 11})
ax2.set_title('Mean Importance Across All Clauses')

plt.suptitle('N-gram Size Decomposition — Phase 5 Champion (40K word 1-3gram)', fontsize=13)
plt.tight_layout()
fig.savefig(RESULTS / 'phase6_anthony_ngram_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: results/phase6_anthony_ngram_decomposition.png")

Saved: results/phase6_anthony_ngram_decomposition.png


## Cell 5: TreeSHAP for HIGH-RISK Clauses
**Why SHAP over split-gain importance:** Split-gain measures how often a feature is used for splitting, but doesn't capture direction or magnitude of contribution to individual predictions. SHAP values decompose each prediction into per-feature contributions, so we can compare what drives TRUE POSITIVES vs FALSE POSITIVES.

In [5]:
import shap

shap_results = {}
for clause in hr_valid:
    j = valid_clauses.index(clause)
    m = lgbm_models[j]
    if m is None:
        print(f"  [{clause}] — no model (skipped)")
        continue
    
    explainer = shap.TreeExplainer(m)
    shap_vals = explainer.shap_values(X_test)
    
    # For binary classifier, shap_values returns [class0, class1] or just array
    if isinstance(shap_vals, list):
        sv = shap_vals[1]  # positive class
    else:
        sv = shap_vals

    # Ensure dense numpy array (SHAP may return sparse or matrix)
    if hasattr(sv, 'toarray'):
        sv = sv.toarray()
    sv = np.asarray(sv)
    if sv.ndim > 2:
        sv = sv.reshape(X_test.shape[0], -1)

    # Mean absolute SHAP per feature (flatten to 1D)
    mean_abs_shap = np.abs(sv).mean(axis=0)
    mean_abs_shap = np.asarray(mean_abs_shap).flatten()
    top_idx = np.argsort(mean_abs_shap)[::-1][:20]
    
    shap_results[clause] = {
        'shap_values': sv,
        'mean_abs': mean_abs_shap,
        'top20_features': [(feature_names[i], float(mean_abs_shap[i])) for i in top_idx],
        'top20_idx': top_idx,
    }
    
    print(f"  [{clause}] top-5 SHAP features:")
    for feat, val in shap_results[clause]['top20_features'][:5]:
        ngram_type = {1: 'uni', 2: 'bi', 3: 'tri'}[len(feat.split())]
        print(f"    {feat:<35} |SHAP|={val:.4f}  ({ngram_type})")

print(f"\nSHAP computed for {len(shap_results)} HIGH-RISK clauses")

  [Uncapped Liability] top-5 SHAP features:
    consequential                       |SHAP|=1.1840  (uni)
    under section                       |SHAP|=0.2690  (bi)
    indirect                            |SHAP|=0.2098  (uni)
    for particular                      |SHAP|=0.1753  (bi)
    group                               |SHAP|=0.1617  (uni)
  [IP Ownership Assignment] top-5 SHAP features:
    title and interest                  |SHAP|=0.3790  (tri)
    intellectual property               |SHAP|=0.3206  (bi)
    hereby assigns                      |SHAP|=0.3094  (bi)
    title and                           |SHAP|=0.2557  (bi)
    no                                  |SHAP|=0.2279  (uni)


  [Change Of Control] top-5 SHAP features:
    change of control                   |SHAP|=0.5223  (tri)
    change of                           |SHAP|=0.3585  (bi)
    to third                            |SHAP|=0.2929  (bi)
    14                                  |SHAP|=0.1909  (uni)
    agents                              |SHAP|=0.1827  (uni)
  [Non-Compete] top-5 SHAP features:
    promote                             |SHAP|=0.2698  (uni)
    directly                            |SHAP|=0.2386  (uni)
    maintain the                        |SHAP|=0.2250  (bi)
    written consent                     |SHAP|=0.1974  (bi)
    written consent of                  |SHAP|=0.1506  (tri)


  [Liquidated Damages] top-5 SHAP features:
    liquidated damages                  |SHAP|=0.4877  (bi)
    to any person                       |SHAP|=0.4572  (tri)
    contractor                          |SHAP|=0.2735  (uni)
    to pay                              |SHAP|=0.2720  (bi)
    security                            |SHAP|=0.2704  (uni)

SHAP computed for 5 HIGH-RISK clauses


## Cell 6: SHAP — True Positive vs False Positive Feature Profiles
**Hypothesis:** False positives are driven by DIFFERENT features than true positives. If we can identify FP-specific features, we can improve precision without hurting recall.

In [6]:
from sklearn.metrics import f1_score

tp_fp_analysis = {}
for clause in hr_valid:
    j = valid_clauses.index(clause)
    m = lgbm_models[j]
    if m is None or clause not in shap_results:
        continue
    
    probs = m.predict_proba(X_test)[:, 1]
    thresh = thresholds[clause]
    preds = (probs >= thresh).astype(int)
    y_true = y_test[:, j]
    
    tp_mask = (preds == 1) & (y_true == 1)
    fp_mask = (preds == 1) & (y_true == 0)
    fn_mask = (preds == 0) & (y_true == 1)
    tn_mask = (preds == 0) & (y_true == 0)
    
    sv = shap_results[clause]['shap_values']
    
    n_tp, n_fp, n_fn, n_tn = tp_mask.sum(), fp_mask.sum(), fn_mask.sum(), tn_mask.sum()
    
    if n_tp < 2 or n_fp < 2:
        print(f"  [{clause}] TP={n_tp}, FP={n_fp} — too few for comparison")
        continue
    
    # Mean SHAP for TP vs FP (flatten to 1D)
    tp_mean_shap = np.asarray(sv[tp_mask].mean(axis=0)).flatten()
    fp_mean_shap = np.asarray(sv[fp_mask].mean(axis=0)).flatten()
    
    # Features that distinguish TP from FP: largest absolute difference
    diff = tp_mean_shap - fp_mean_shap
    diff_idx = np.argsort(np.abs(diff))[::-1][:10]
    
    # Features more active in FP than TP (FP-specific false signals)
    fp_specific_idx = np.argsort(fp_mean_shap)[::-1][:5]
    
    tp_fp_analysis[clause] = {
        'n_tp': int(n_tp), 'n_fp': int(n_fp), 'n_fn': int(n_fn), 'n_tn': int(n_tn),
        'f1': float(f1_score(y_true, preds, zero_division=0)),
        'distinguishing_features': [
            (feature_names[i], float(tp_mean_shap[i]), float(fp_mean_shap[i]), float(diff[i]))
            for i in diff_idx
        ],
        'fp_specific_features': [
            (feature_names[i], float(fp_mean_shap[i]))
            for i in fp_specific_idx
        ],
    }
    
    print(f"\n  [{clause}] TP={n_tp} FP={n_fp} FN={n_fn} F1={tp_fp_analysis[clause]['f1']:.3f}")
    print(f"  Features that DISTINGUISH TP from FP (largest SHAP gap):")
    print(f"  {'Feature':<35} {'TP SHAP':>9} {'FP SHAP':>9} {'Gap':>8}")
    for feat, tp_s, fp_s, gap in tp_fp_analysis[clause]['distinguishing_features'][:5]:
        direction = '→TP' if gap > 0 else '→FP'
        print(f"  {feat:<35} {tp_s:>+9.4f} {fp_s:>+9.4f} {gap:>+7.4f} {direction}")
    
    print(f"  FP-specific triggers (positive SHAP in false positives):")
    for feat, val in tp_fp_analysis[clause]['fp_specific_features'][:3]:
        print(f"    {feat:<35} FP SHAP={val:+.4f}")


  [Uncapped Liability] TP=19 FP=17 FN=5 F1=0.633
  Features that DISTINGUISH TP from FP (largest SHAP gap):
  Feature                               TP SHAP   FP SHAP      Gap
  consequential                         +1.3610   +1.4989 -0.1379 →FP
  party be liable                       +0.1482   +0.0168 +0.1314 →TP
  audit                                 +0.0382   +0.1474 -0.1092 →FP
  foregoing                             +0.1397   +0.0453 +0.0944 →TP
  federal state                         +0.0228   -0.0555 +0.0783 →TP
  FP-specific triggers (positive SHAP in false positives):
    consequential                       FP SHAP=+1.4989
    under section                       FP SHAP=+0.2251
    be liable                           FP SHAP=+0.1945

  [IP Ownership Assignment] TP=16 FP=9 FN=7 F1=0.667
  Features that DISTINGUISH TP from FP (largest SHAP gap):
  Feature                               TP SHAP   FP SHAP      Gap
  hereby assigns                        +0.4753   -0.2054 +0.6807 →


  [Non-Compete] TP=17 FP=22 FN=8 F1=0.531
  Features that DISTINGUISH TP from FP (largest SHAP gap):
  Feature                               TP SHAP   FP SHAP      Gap
  competition                           +0.0661   -0.0309 +0.0969 →TP
  storage                               -0.0532   +0.0425 -0.0957 →FP
  exclusive right                       -0.0006   +0.0944 -0.0951 →FP
  competitive with                      +0.0629   -0.0100 +0.0729 →TP
  then such                             +0.0774   +0.0055 +0.0720 →TP
  FP-specific triggers (positive SHAP in false positives):
    directly                            FP SHAP=+0.1563
    promote                             FP SHAP=+0.1449
    maintain the                        FP SHAP=+0.1310



  [Liquidated Damages] TP=7 FP=6 FN=7 F1=0.519
  Features that DISTINGUISH TP from FP (largest SHAP gap):
  Feature                               TP SHAP   FP SHAP      Gap
  liquidated damages                    +2.6737   -0.2365 +2.9103 →TP
  amounts paid                          -0.0152   +0.1651 -0.1803 →FP
  names and                             -0.1048   +0.0693 -0.1742 →FP
  contractor                            +0.1278   +0.2979 -0.1701 →FP
  omission                              -0.0435   +0.1131 -0.1566 →FP
  FP-specific triggers (positive SHAP in false positives):
    to terminate                        FP SHAP=+0.3615
    contractor                          FP SHAP=+0.2979
    to pay                              FP SHAP=+0.2648


## Cell 7: SHAP Summary Visualization for High-Risk Clauses

In [7]:
n_hr = len([c for c in hr_valid if c in shap_results])
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, clause in enumerate(hr_valid):
    if clause not in shap_results or idx >= 6:
        break
    ax = axes[idx]
    top_features = shap_results[clause]['top20_features'][:12]
    names = [f[:30] for f, _ in top_features]
    vals = [v for _, v in top_features]
    
    # Color by n-gram size
    colors = []
    for f, _ in top_features:
        n = len(f.split())
        colors.append('#1565c0' if n == 1 else '#42a5f5' if n == 2 else '#90caf9')
    
    j = valid_clauses.index(clause)
    f1_val = float(f1_score(y_test[:, j], 
                            (lgbm_models[j].predict_proba(X_test)[:, 1] >= thresholds[clause]).astype(int),
                            zero_division=0))
    
    y_pos = range(len(names) - 1, -1, -1)
    ax.barh(list(y_pos), vals, color=colors)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_xlabel('Mean |SHAP|')
    ax.set_title(f'{clause}\n(F1={f1_val:.3f})', fontsize=10)

# Hide unused axes
for i in range(len(hr_valid), 6):
    axes[i].axis('off')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1565c0', label='Unigram'),
                   Patch(facecolor='#42a5f5', label='Bigram'),
                   Patch(facecolor='#90caf9', label='Trigram')]
fig.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.suptitle('TreeSHAP Top Features per HIGH-RISK Clause — Phase 5 Champion (LGBM + 40K trigrams)', fontsize=13)
plt.tight_layout()
fig.savefig(RESULTS / 'phase6_anthony_shap_hr_clauses.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: results/phase6_anthony_shap_hr_clauses.png")

Saved: results/phase6_anthony_shap_hr_clauses.png


## Cell 8: Cross-Clause Feature Sharing
**Question:** Do different clause detectors share features, or are they independent? If clauses share features, it explains why multi-label interactions matter. If independent, each clause is essentially a separate binary classifier.

In [8]:
# Build binary feature-usage matrix: which features does each clause use?
used_matrix = np.zeros((n_clauses, n_features), dtype=bool)
for j, clause in enumerate(valid_clauses):
    m = lgbm_models[j]
    if m is None:
        continue
    used_matrix[j] = m.feature_importances_ > 0

# Jaccard similarity between clause feature sets
active_clauses = [j for j in range(n_clauses) if lgbm_models[j] is not None]
n_active = len(active_clauses)
jaccard = np.zeros((n_active, n_active))
for i_idx, i in enumerate(active_clauses):
    for j_idx, j_c in enumerate(active_clauses):
        inter = np.sum(used_matrix[i] & used_matrix[j_c])
        union = np.sum(used_matrix[i] | used_matrix[j_c])
        jaccard[i_idx, j_idx] = inter / max(1, union)

# Summary
np.fill_diagonal(jaccard, np.nan)
mean_jaccard = np.nanmean(jaccard)
max_pair = np.unravel_index(np.nanargmax(jaccard), jaccard.shape)
max_jaccard = jaccard[max_pair]

print(f"Cross-clause feature sharing (Jaccard similarity):")
print(f"  Mean pairwise Jaccard: {mean_jaccard:.3f}")
print(f"  Most similar pair: {valid_clauses[active_clauses[max_pair[0]]]} ↔ {valid_clauses[active_clauses[max_pair[1]]]} (J={max_jaccard:.3f})")

# Features used by many clauses
usage_count = used_matrix[active_clauses].sum(axis=0)
shared_10plus = np.sum(usage_count >= 10)
shared_20plus = np.sum(usage_count >= 20)
unique_to_one = np.sum(usage_count == 1)

print(f"\nFeature sharing profile:")
print(f"  Used by 20+ clauses: {shared_20plus} features")
print(f"  Used by 10+ clauses: {shared_10plus} features")
print(f"  Used by exactly 1 clause: {unique_to_one} features")

# Top shared features
top_shared_idx = np.argsort(usage_count)[::-1][:10]
print(f"\nMost shared features (used by N clauses):")
for i in top_shared_idx:
    if usage_count[i] > 0:
        print(f"  {feature_names[i]:<35} used by {usage_count[i]} clauses")

Cross-clause feature sharing (Jaccard similarity):
  Mean pairwise Jaccard: 0.025
  Most similar pair: Governing Law ↔ Anti-Assignment (J=0.072)

Feature sharing profile:
  Used by 20+ clauses: 0 features
  Used by 10+ clauses: 2 features
  Used by exactly 1 clause: 3292 features

Most shared features (used by N clauses):
  10                                  used by 13 clauses
  cause                               used by 10 clauses
  against                             used by 9 clauses
  granted                             used by 9 clauses
  required                            used by 8 clauses
  either party                        used by 8 clauses
  deemed                              used by 8 clauses
  and conditions                      used by 8 clauses
  this agreement                      used by 8 clauses
  license                             used by 8 clauses


## Cell 9: Cross-Clause Heatmap Visualization

In [9]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(14, 12))

active_names = [valid_clauses[j] for j in active_clauses]
# Reset diagonal for display
jaccard_display = jaccard.copy()
np.fill_diagonal(jaccard_display, 1.0)

sns.heatmap(jaccard_display, xticklabels=active_names, yticklabels=active_names,
            cmap='YlOrRd', vmin=0, vmax=0.5, annot=False, ax=ax,
            linewidths=0.3, linecolor='white')
ax.set_title('Cross-Clause Feature Sharing (Jaccard Similarity)\nLow = independent detectors | High = shared signal', fontsize=12)
ax.set_xticklabels(active_names, rotation=90, fontsize=7)
ax.set_yticklabels(active_names, rotation=0, fontsize=7)

plt.tight_layout()
fig.savefig(RESULTS / 'phase6_anthony_cross_clause_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: results/phase6_anthony_cross_clause_heatmap.png")

Saved: results/phase6_anthony_cross_clause_heatmap.png


## Cell 10: SHAP vs Split-Gain Importance Comparison
**Question:** Do SHAP and split-gain agree? Mark used split-gain in his analysis. If they disagree, SHAP is more trustworthy (model-faithful), and Mark's conclusions may need revision.

In [10]:
from scipy.stats import spearmanr

print(f"SHAP vs Split-Gain Rank Correlation per HIGH-RISK Clause:")
print(f"{'Clause':<30} {'Spearman ρ':>10} {'Top-10 overlap':>15} {'Interpretation':>20}")
print('-' * 78)

shap_vs_gain = {}
for clause in hr_valid:
    if clause not in shap_results:
        continue
    j = valid_clauses.index(clause)
    m = lgbm_models[j]
    if m is None:
        continue
    
    gain_imp = np.asarray(m.feature_importances_).flatten()
    shap_imp = np.asarray(shap_results[clause]['mean_abs']).flatten()
    
    # Only compare features that are non-zero in at least one
    active_mask = (gain_imp > 0) | (shap_imp > 0)
    if active_mask.sum() < 10:
        continue
    
    rho, pval = spearmanr(gain_imp[active_mask], shap_imp[active_mask])
    
    # Top-10 overlap
    gain_top10 = set(np.argsort(gain_imp)[::-1][:10])
    shap_top10 = set(np.argsort(shap_imp)[::-1][:10])
    overlap = len(gain_top10 & shap_top10)
    
    interp = 'Strong agree' if rho > 0.8 else 'Moderate agree' if rho > 0.5 else 'DISAGREE'
    shap_vs_gain[clause] = {'rho': round(float(rho), 3), 'top10_overlap': overlap}
    
    print(f"{clause:<30} {rho:>9.3f} {overlap:>10}/10 {interp:>20}")

SHAP vs Split-Gain Rank Correlation per HIGH-RISK Clause:
Clause                         Spearman ρ  Top-10 overlap       Interpretation
------------------------------------------------------------------------------
Uncapped Liability                 0.512          6/10       Moderate agree
IP Ownership Assignment            0.559          5/10       Moderate agree
Change Of Control                  0.573          8/10       Moderate agree
Non-Compete                        0.604          7/10       Moderate agree
Liquidated Damages                 0.490          8/10             DISAGREE


## Cell 11: Domain Validation — Do SHAP Top Features Match Legal Expectations?
CUAD annotation guidelines define what each clause category means legally. If SHAP top features are legally meaningful terms (not proxy features), the model is reasoning correctly.

In [11]:
domain_terms = {
    'Uncapped Liability': [
        'unlimited', 'without limit', 'uncapped', 'no cap', 'no limitation',
        'not limited', 'all damages', 'consequential', 'indirect',
    ],
    'IP Ownership Assignment': [
        'intellectual property', 'assign', 'ownership', 'work product',
        'work made for hire', 'rights title', 'proprietary',
    ],
    'Change Of Control': [
        'change of control', 'acquisition', 'merger', 'ownership',
        'voting', 'majority', 'controlling interest',
    ],
    'Non-Compete': [
        'non compete', 'noncompete', 'compete', 'competitive',
        'restrictive covenant', 'territory', 'business activities',
    ],
    'Liquidated Damages': [
        'liquidated damages', 'penalty', 'per day', 'specified amount',
        'pre agreed', 'predetermined',
    ],
    'Joint IP Ownership': [
        'joint', 'jointly owned', 'joint ownership', 'co owned',
        'intellectual property', 'joint invention',
    ],
}

print(f"Domain Validation — SHAP top-20 vs expected legal terms:")
print(f"{'Clause':<30} {'Matched':>8} {'Expected':>9} {'Coverage':>9} {'Verdict':>15}")
print('-' * 74)

domain_validation = {}
for clause in hr_valid:
    if clause not in shap_results:
        continue
    top_feats = [f.lower() for f, _ in shap_results[clause]['top20_features']]
    expected = domain_terms.get(clause, [])
    if not expected:
        continue
    
    matched = []
    for term in expected:
        for feat in top_feats:
            if term in feat or feat in term:
                matched.append(term)
                break
    
    coverage = len(matched) / len(expected)
    verdict = 'VALIDATED' if coverage >= 0.5 else 'PARTIAL' if coverage >= 0.2 else 'PROXY'
    domain_validation[clause] = {
        'matched': matched,
        'n_matched': len(matched),
        'n_expected': len(expected),
        'coverage': round(coverage, 3),
        'verdict': verdict,
    }
    print(f"{clause:<30} {len(matched):>5}/{len(expected):<3} {'':>5} {100*coverage:>7.0f}% {verdict:>15}")
    if matched:
        print(f"  Matched terms: {matched}")

Domain Validation — SHAP top-20 vs expected legal terms:
Clause                          Matched  Expected  Coverage         Verdict
--------------------------------------------------------------------------
Uncapped Liability                 2/9              22%         PARTIAL
  Matched terms: ['consequential', 'indirect']
IP Ownership Assignment            3/7              43%         PARTIAL
  Matched terms: ['intellectual property', 'assign', 'work made for hire']
Change Of Control                  3/7              43%         PARTIAL
  Matched terms: ['change of control', 'ownership', 'voting']
Non-Compete                        3/7              43%         PARTIAL
  Matched terms: ['non compete', 'noncompete', 'compete']
Liquidated Damages                 1/6              17%           PROXY
  Matched terms: ['liquidated damages']


## Cell 12: Consolidate Findings + Save Results

In [12]:
results = {
    'date': '2026-04-19',
    'phase': '6 (Explainability)',
    'author': 'Anthony Rodrigues',
    'model': 'Phase 5 champion: LGBM-only + 40K word 1-3gram + class-prior thresholds',
    'feature_utilization': {
        'mean_features_per_clause': round(np.mean(used_counts)),
        'max_features_per_clause': int(np.max(used_counts)),
        'min_features_per_clause': int(np.min(used_counts)),
        'union_all_clauses': int(len(all_used_features)),
        'total_features': n_features,
        'pct_used_union': round(100 * len(all_used_features) / n_features, 1),
    },
    'ngram_decomposition': {
        'mean_unigram_pct': round(mean_uni, 1),
        'mean_bigram_pct': round(mean_bi, 1),
        'mean_trigram_pct': round(mean_tri, 1),
        'per_hr_clause': {c: ngram_importance[c] for c in hr_valid if c in ngram_importance},
    },
    'shap_top_features': {
        clause: [(f, round(v, 4)) for f, v in shap_results[clause]['top20_features'][:10]]
        for clause in hr_valid if clause in shap_results
    },
    'tp_fp_analysis': {
        clause: {
            'n_tp': info['n_tp'], 'n_fp': info['n_fp'], 'n_fn': info['n_fn'],
            'f1': info['f1'],
            'top3_distinguishing': [
                {'feature': f, 'tp_shap': round(tp, 4), 'fp_shap': round(fp, 4), 'gap': round(g, 4)}
                for f, tp, fp, g in info['distinguishing_features'][:3]
            ]
        }
        for clause, info in tp_fp_analysis.items()
    },
    'cross_clause': {
        'mean_jaccard': round(float(mean_jaccard), 3),
        'most_similar_pair': f"{valid_clauses[active_clauses[max_pair[0]]]} ↔ {valid_clauses[active_clauses[max_pair[1]]]}",
        'most_similar_jaccard': round(float(max_jaccard), 3),
    },
    'shap_vs_gain': shap_vs_gain,
    'domain_validation': domain_validation,
}

with open(RESULTS / 'phase6_anthony_explainability.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print("Results saved to results/phase6_anthony_explainability.json")
print()
print("=" * 72)
print("KEY FINDINGS")
print("=" * 72)
print(f"")
print(f"1. FEATURE UTILIZATION: Mean {np.mean(used_counts):.0f}/{n_features:,} features used per clause ({100*np.mean(used_counts)/n_features:.1f}%).")
print(f"   Union across all clauses: {len(all_used_features):,}/{n_features:,} ({100*len(all_used_features)/n_features:.1f}%).")
print(f"   → {n_features - len(all_used_features):,} features ({100*(n_features - len(all_used_features))/n_features:.1f}%) are NEVER used.")
print(f"")
print(f"2. N-GRAM DECOMPOSITION: Unigrams={mean_uni:.1f}%, Bigrams={mean_bi:.1f}%, Trigrams={mean_tri:.1f}%.")
if mean_tri > 5:
    print(f"   → Trigrams carry meaningful signal ({mean_tri:.1f}%). Phase 4 trigram expansion was justified.")
else:
    print(f"   → Trigrams carry minimal signal ({mean_tri:.1f}%). The real gain from Phase 4 was more bigrams.")
print(f"")
print(f"3. CROSS-CLAUSE SHARING: Mean Jaccard={mean_jaccard:.3f}.")
if mean_jaccard < 0.15:
    print(f"   → Clause detectors are mostly INDEPENDENT. Each clause uses its own feature set.")
else:
    print(f"   → Moderate feature sharing. Some clauses use common legal vocabulary.")
print(f"")
print(f"4. DOMAIN VALIDATION:")
for clause in hr_valid:
    if clause in domain_validation:
        dv = domain_validation[clause]
        print(f"   {clause}: {dv['n_matched']}/{dv['n_expected']} expected terms found → {dv['verdict']}")

Results saved to results/phase6_anthony_explainability.json

KEY FINDINGS

1. FEATURE UTILIZATION: Mean 277/40,000 features used per clause (0.7%).
   Union across all clauses: 4,886/40,000 (12.2%).
   → 35,114 features (87.8%) are NEVER used.

2. N-GRAM DECOMPOSITION: Unigrams=39.0%, Bigrams=42.1%, Trigrams=18.9%.
   → Trigrams carry meaningful signal (18.9%). Phase 4 trigram expansion was justified.

3. CROSS-CLAUSE SHARING: Mean Jaccard=0.025.
   → Clause detectors are mostly INDEPENDENT. Each clause uses its own feature set.

4. DOMAIN VALIDATION:
   Uncapped Liability: 2/9 expected terms found → PARTIAL
   IP Ownership Assignment: 3/7 expected terms found → PARTIAL
   Change Of Control: 3/7 expected terms found → PARTIAL
   Non-Compete: 3/7 expected terms found → PARTIAL
   Liquidated Damages: 1/6 expected terms found → PROXY
